# 05_marian_baseline_eval

In [2]:
from transformers import MarianMTModel, MarianTokenizer
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
from datasets import load_from_disk

dataset = load_from_disk("../data/processed/iitb_filtered")

In [4]:
test_subset = dataset["test"].shuffle(seed=42).select(range(min(10_000, len(dataset["test"]))))

In [5]:
en_hi_tokenizer = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-hi")
en_hi_model = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-en-hi").to(device)

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

d:\Dl projects\mt-en-hi\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kunja\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B /  306MB            

model.safetensors: downloading bytes:           |  0.00B            

In [6]:
predictions = []
references = []

for i, sentence in enumerate(test_subset["en"]):
    inputs = en_hi_tokenizer(sentence, return_tensors="pt", padding=True, truncation=True).to(device)
    translated = en_hi_model.generate(**inputs, max_length=80, num_beams=4, early_stopping=True)
    translated_text = en_hi_tokenizer.batch_decode(translated, skip_special_tokens=True)[0]
    predictions.append(translated_text)
    references.append(test_subset["hi"][i])

In [7]:
predictions[:5], references[:5]

(['उन्होंने कभी विश्वास नहीं किया कि वह जिस तरह से प्रधान की मृत्यु हुई.',
  'पिछले साल से लेकर 20 से 30 प्रतिशत लोगों की गिनती बढ़कर 20 प्रतिशत हो गयी है ।',
  'इसके अलावा, कई लोगों को लगता है कि माल खरीदने के लिए पैसा ही काफी होता है ।',
  'पाँच बच्चे रामिकला थे, नटवर लील, डब्बू और दो बहनें, ट्रिक और यामकी ।',
  'उत्तर सागर की गर्मी से सुरक्षा पाने के लिए पक्षी फिर से जंगली बन गए'],
 ['उन्हें शेरिफ के बताये गये तरीके से मृत्यु पर बिलकुल भी भरोसा नहीं है।',
  'ऐसी खरीदारी में बीते वर्ष की अपेक्षा 20 से 30 फीसदी बढ़ोतरी हुई है।',
  'वॉल स्ट्रीट पर  बैगेज की फीस एक संकेत के रूप में कई देख रहे हैं कि कई एयरलाइन घाटे के वर्षों के बाद हवाई यात्रा की लागत को कवर करने के लिए पर्याप्त पैसे चार्ज कर रहे हैं.',
  'परिवार की पांच संतानों में रमणिकलाल, नटवर लाल, धीरूभाई और दो बहनें त्रिलोचना और जसुमती शामिल हैं।',
  'नॉर्थ सी रिग से सुरक्षा के लिए पक्षी को हवाई परिवहन द्वारा वापस जंगल में छोड़ा गया'])

In [8]:
import sacrebleu

bleu = sacrebleu.corpus_bleu(predictions, [references])
print(f"BLEU score: {bleu.score:.2f}")

BLEU score: 9.85


In [9]:
hi_en_tokenizer = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-hi-en")
hi_en_model = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-hi-en").to(device)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  304MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

d:\Dl projects\mt-en-hi\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kunja\.cache\huggingface\hub\models--Helsinki-NLP--opus-mt-hi-en. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  304MB            

model.safetensors: downloading bytes:           |  0.00B            

In [10]:
predictions_hi = []
references_hi = []

for i, sentence in enumerate(test_subset["hi"]):
    inputs = hi_en_tokenizer(sentence, return_tensors="pt", padding=True, truncation=True).to(device)
    translated = hi_en_model.generate(**inputs, max_length=80, num_beams=4, early_stopping=True)
    translated_text = hi_en_tokenizer.batch_decode(translated, skip_special_tokens=True)[0]
    predictions_hi.append(translated_text)
    references_hi.append(test_subset["en"][i])

In [11]:
import sacrebleu

bleu_hi = sacrebleu.corpus_bleu(predictions_hi, [references_hi])
print(f"BLEU score: {bleu_hi.score:.2f}")

BLEU score: 13.43


In [13]:
import json

marian_baseline_results = {
    "model": "MarianMT (Helsinki-NLP opus-mt, zero-shot, no fine-tuning)",
    "en_hi": {
        "bleu_score": 9.85,
    },
    "hi_en": {
        "bleu_score": 13.42,
    }
}

with open("../results/marian_baseline_results.json", "w") as f:
    json.dump(marian_baseline_results, f, indent=2, ensure_ascii=False)